# Base Test: 안건별 병합과 검증 전후 비교

TS3003c에 대한 PR #35의 기생성 핵심 구간을 사용한다. 발화가 1,000자를 넘으면 보존하고, 긴 구간은 발화 경계에서 추가 분할한다. 검증 전후를 저장하며, LLM 검증은 사람의 평가를 대체하지 않는다. 결과와 사용량은 `outputs/`에 저장된다.

In [ ]:
from pathlib import Path
import os, sys, json
root = Path.cwd()
if not (root / 'experiment.py').exists():
    root = root / 'playground' / 'hyungjin' / 'base-test'
sys.path.insert(0, str(root.resolve()))
from experiment import Experiment, prepare
data, chunks = prepare()
print('회의:', data['meeting_id'], '구간:', len(data['segments']), '추출용 청크:', len(chunks))
print('모델: gpt-6-luna / low')
print('첫 청크:', json.dumps(chunks[0], ensure_ascii=False, indent=2)[:1500])

## 1. 실행 준비
실행 시 전사문이 OpenAI API에 전달되며 사용량이 발생한다. 새 실행 폴더를 만들고 설정·입력·프롬프트를 저장한다.

In [ ]:
exp = Experiment()
print('저장 위치:', exp.output)

## 2. 청크별 안건·논의 및 독립 액션 추출
같은 청크와 기존 안건 목록을 두 에이전트에 각각 전달한다. 신규 안건의 액션 연결은 미확정일 수 있다.

In [ ]:
agendas = exp.run_chunks()
print(json.dumps(agendas, ensure_ascii=False, indent=2))

## 3. 안건별 1차 병합과 검증
병합 결과를 저장한 뒤 원 청크로 확인하고, 오류가 있으면 한 번 수정하여 다시 검증한다. 재검증의 미해결 항목을 확인해야 한다.

In [ ]:
before, after = exp.merge_and_review()
print('안건 수:', len(before))

## 4. 검증 전후 비교
검증 모델의 발견 건수는 확정 오류 수가 아니다. `04_병합전검증`, `05_검증`, `06_수정후`, `07_재검증`과 원문을 함께 검토한다.

In [ ]:
for b, a, v in zip(exp.before, exp.after, exp.reviews):
    print('\n---', b.get('agenda_id'), b.get('title'), '---')
    print('검증 의견:', json.dumps(v.get('issues', []), ensure_ascii=False, indent=2))
    print('검증 전:', json.dumps(b, ensure_ascii=False, indent=2))
    print('수정 후:', json.dumps(a, ensure_ascii=False, indent=2))
print('\n실행 기록:', exp.output / '실행정보.json')